In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  # Native PyTorch AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# Set seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 1. Load Data
print("1. Loading datasets...")
train_df = pd.read_csv('train[1].csv')
test_df = pd.read_csv('test[1].csv')

# Handle NaNs across all columns
fill_dict = {
    'summary': '', 'positives': '', 'negatives': '', 'advice_to_mgmt': '',
    'job_title': 'N/A', 'location': 'N/A', 'status': 'N/A',
    'score_1': 3, 'score_2': 3, 'score_3': 3, 'score_4': 3, 'score_5': 3
}
train_df = train_df.fillna(fill_dict)
test_df = test_df.fillna(fill_dict)

# Map target 1-5 to 0-4 for PyTorch classification
train_df['target'] = train_df['overall'] - 1

# 2. Convert Tabular + Text into Structured Prompts
def create_text_prompts(df):
    prompts = []
    for _, row in df.iterrows():
        # Critical text FIRST so it never gets truncated
        prompt = (
            f"NEGATIVES: {row['negatives']}. "
            f"POSITIVES: {row['positives']}. "
            f"SUMMARY: {row['summary']}. "
            f"ADVICE: {row['advice_to_mgmt']}. "
            f"SCORES: {row['score_1']}/5, {row['score_2']}/5, {row['score_3']}/5, {row['score_4']}/5, {row['score_5']}/5. "
            f"TITLE: {row['job_title']}."
        )
        prompts.append(prompt)
    return prompts

print("2. Generating structured textual prompts...")
train_df['prompt'] = create_text_prompts(train_df)
test_df['prompt'] = create_text_prompts(test_df)

# Train / Validation Split
train_data, val_data = train_test_split(
    train_df, test_size=0.15, random_state=SEED, stratify=train_df['target']
)

# 3. PyTorch Dataset Class
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=256):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = ReviewDataset(train_data['prompt'].values, train_data['target'].values)
val_dataset = ReviewDataset(val_data['prompt'].values, val_data['target'].values)
test_dataset = ReviewDataset(test_df['prompt'].values)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 4. Initialize Model, Optimizer, Scheduler
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5
)
model.to(device)

EPOCHS = 8
optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

# 5. Training Loop
print("3. Training Transformer Model...")

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        if scaler:
            with torch.amp.autocast('cuda'):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

        scheduler.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # Validation Phase
    model.eval()
    val_preds, val_targets = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(labels.cpu().numpy())

    val_f1 = f1_score(val_targets, val_preds, average='macro')
    val_acc = accuracy_score(val_targets, val_preds)

    print(f"\n---> Epoch {epoch + 1} | Train Loss: {avg_train_loss:.4f} | Val Macro F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}\n")

# 6. Generate Predictions on Test Set
print("4. Generating predictions on test dataset...")
model.eval()
test_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1).cpu().numpy()
        test_preds.extend(preds)

# Shift predicted targets (0-4) back to original 1-5 rating scale
final_ratings = np.array(test_preds) + 1

# 7. Write submission.csv
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'overall': final_ratings
})

submission.to_csv('submission.csv', index=False)
print("\nSuccess! Transformer file 'submission.csv' generated.")
print(submission.head())

Using device: cuda
1. Loading datasets...
2. Generating structured textual prompts...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...


---> Epoch 7 | Train Loss: 0.8033 | Val Macro F1: 0.3377 | Val Acc: 0.3826



Epoch 8/8: 100%|██████████| 645/645 [02:25<00:00,  4.44it/s]



---> Epoch 8 | Train Loss: 0.7205 | Val Macro F1: 0.3409 | Val Acc: 0.3873

4. Generating predictions on test dataset...


Inference: 100%|██████████| 95/95 [00:54<00:00,  1.74it/s]


Success! Transformer file 'submission.csv' generated.
      ID  overall
0  24788        3
1   6607        5
2  13591        3
3  35601        3
4  35466        5


In [ ]:
import pandas as pd
import numpy as np
import scipy as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score
from lightgbm import LGBMRegressor
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. Threshold Optimizer class for Macro F1 Optimization
# ---------------------------------------------------------
class MacroF1ThresholdOptimizer:
    def __init__(self):
        self.coef_ = [1.5, 2.5, 3.5, 4.5]

    def _loss(self, coef, X, y):
        X_p = np.copy(X)
        preds = np.zeros(len(X_p))
        preds[X_p < coef[0]] = 1
        preds[(X_p >= coef[0]) & (X_p < coef[1])] = 2
        preds[(X_p >= coef[1]) & (X_p < coef[2])] = 3
        preds[(X_p >= coef[2]) & (X_p < coef[3])] = 4
        preds[X_p >= coef[3]] = 5

        # Negative Macro F1 (since scipy minimizes)
        return -f1_score(y, preds, average='macro')

    def fit(self, X, y):
        initial_coef = [1.5, 2.5, 3.5, 4.5]
        res = sp.optimize.minimize(self._loss, initial_coef, args=(X, y), method='Nelder-Mead')
        self.coef_ = res.x
        print(f"  Optimized Cutoff Thresholds: {np.round(self.coef_, 3)}")

    def predict(self, X):
        X_p = np.copy(X)
        preds = np.zeros(len(X_p))
        preds[X_p < self.coef_[0]] = 1
        preds[(X_p >= self.coef_[0]) & (X_p < self.coef_[1])] = 2
        preds[(X_p >= self.coef_[1]) & (X_p < self.coef_[2])] = 3
        preds[(X_p >= self.coef_[2]) & (X_p < self.coef_[3])] = 4
        preds[X_p >= self.coef_[3]] = 5
        return preds.astype(int)

# ---------------------------------------------------------
# 2. Pipeline Execution
# ---------------------------------------------------------
def main():
    print("1. Loading datasets...")
    train = pd.read_csv('train[1].csv')
    test = pd.read_csv('test[1].csv')

    target = train['overall'].values  # 1 to 5 continuous target

    print("2. Engineering Features...")
    score_cols = [f'score_{i}' for i in range(1, 6)]

    for df in [train, test]:
        for col in score_cols:
            df[col] = df[col].fillna(df[col].median())
        df['score_mean'] = df[score_cols].mean(axis=1)
        df['score_std'] = df[score_cols].std(axis=1).fillna(0)
        df['score_min'] = df[score_cols].min(axis=1)
        df['score_max'] = df[score_cols].max(axis=1)

        text_cols = ['summary', 'positives', 'negatives', 'advice_to_mgmt']
        for col in text_cols:
            df[col] = df[col].fillna('').astype(str)

    print("3. Extracting Text TF-IDF Features...")
    preprocessor = ColumnTransformer(
        transformers=[
            ('summary_tf', TfidfVectorizer(max_features=1000, ngram_range=(1,2), stop_words='english'), 'summary'),
            ('pos_tf', TfidfVectorizer(max_features=1500, ngram_range=(1,2), stop_words='english'), 'positives'),
            ('neg_tf', TfidfVectorizer(max_features=1500, ngram_range=(1,2), stop_words='english'), 'negatives'),
            ('adv_tf', TfidfVectorizer(max_features=1000, stop_words='english'), 'advice_to_mgmt')
        ],
        remainder='drop'
    )

    X_train_text = preprocessor.fit_transform(train).toarray()
    X_test_text = preprocessor.transform(test).toarray()

    numeric_cols = score_cols + ['score_mean', 'score_std', 'score_min', 'score_max']
    X_train = np.hstack([train[numeric_cols].values, X_train_text])
    X_test = np.hstack([test[numeric_cols].values, X_test_text])

    print("4. Training LGBM Regressor with 5-Fold Stratified CV...")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    oof_predictions = np.zeros(len(train))
    test_predictions = np.zeros(len(test))

    lgb_params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.03,
        'num_leaves': 31,
        'n_estimators': 800,
        'random_state': 42,
        'verbosity': -1
    }

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, target)):
        X_tr, y_tr = X_train[train_idx], target[train_idx]
        X_va, y_va = X_train[val_idx], target[val_idx]

        model = LGBMRegressor(**lgb_params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])

        oof_predictions[val_idx] = model.predict(X_va)
        test_predictions += model.predict(X_test) / 5.0

    print("\n5. Optimizing Class Decision Thresholds for Macro F1...")
    optimizer = MacroF1ThresholdOptimizer()
    optimizer.fit(oof_predictions, target)

    oof_discrete = optimizer.predict(oof_predictions)
    cv_f1 = f1_score(target, oof_discrete, average='macro')
    print(f"\n==========================================")
    print(f"  --> Final Out-Of-Fold Macro F1: {cv_f1:.4f}")
    print(f"==========================================\n")

    print("6. Writing submission.csv...")
    final_test_preds = optimizer.predict(test_predictions)

    submission = pd.DataFrame({
        'ID': test['ID'],
        'overall': final_test_preds
    })

    submission.to_csv('submission.csv', index=False)
    print("Success! File 'submission.csv' generated.")
    print(submission.head())

if __name__ == "__main__":
    main()

1. Loading datasets...
2. Engineering Features...
3. Extracting Text TF-IDF Features...
4. Training LGBM Regressor with 5-Fold Stratified CV...

5. Optimizing Class Decision Thresholds for Macro F1...
  Optimized Cutoff Thresholds: [1.595 2.875 3.537 4.092]

  --> Final Out-Of-Fold Macro F1: 0.3347

6. Writing submission.csv...
Success! File 'submission.csv' generated.
      ID  overall
0  24788        3
1   6607        5
2  13591        3
3  35601        3
4  35466        5


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. Setup & Data Loading
# ---------------------------------------------------------
SEED = 42
MODEL_NAME = 'distilbert-base-uncased'

print("1. Loading datasets...")
train_df = pd.read_csv('train[1].csv')
test_df = pd.read_csv('test[1].csv')

fill_dict = {
    'summary': '', 'positives': '', 'negatives': '', 'advice_to_mgmt': '',
    'job_title': 'N/A', 'location': 'N/A', 'status': 'N/A',
    'score_1': '3', 'score_2': '3', 'score_3': '3', 'score_4': '3', 'score_5': '3'
}
train_df = train_df.fillna(fill_dict)
test_df = test_df.fillna(fill_dict)

# Map labels 1-5 to 0-4 for standard PyTorch classification
train_df['label'] = train_df['overall'] - 1

def create_structured_prompt(df):
    prompts = []
    for _, r in df.iterrows():
        prompt = (
            f"NEGATIVES: {r['negatives']} | "
            f"POSITIVES: {r['positives']} | "
            f"SUMMARY: {r['summary']} | "
            f"ADVICE: {r['advice_to_mgmt']} | "
            f"RATINGS: Work={r['score_1']}/5, Culture={r['score_2']}/5, Mgmt={r['score_3']}/5, Comp={r['score_4']}/5, Opps={r['score_5']}/5 | "
            f"ROLE: {r['job_title']}"
        )
        prompts.append(prompt)
    return prompts

print("2. Constructing structured prompts...")
train_df['text'] = create_structured_prompt(train_df)
test_df['text'] = create_structured_prompt(test_df)

# Train/Val Split
train_data, val_data = train_test_split(
    train_df, test_size=0.15, random_state=SEED, stratify=train_df['label']
)

# ---------------------------------------------------------
# 2. Compute Class Weights (Balanced Loss)
# ---------------------------------------------------------
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_data['label'].values),
    y=train_data['label'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print(f"Calculated Class Weights: {np.round(class_weights, 2)}")

# ---------------------------------------------------------
# 3. Tokenization using Hugging Face Datasets
# ---------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=320
    )

print("3. Tokenizing datasets...")
hf_train = Dataset.from_pandas(train_data[['text', 'label']])
hf_val = Dataset.from_pandas(val_data[['text', 'label']])
hf_test = Dataset.from_pandas(test_df[['text', 'ID']])

tokenized_train = hf_train.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_val = hf_val.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_test = hf_test.map(tokenize_function, batched=True, remove_columns=['text', 'ID'])

# ---------------------------------------------------------
# 4. Custom Trainer with Class Weighted Loss
# ---------------------------------------------------------
class CustomWeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# ---------------------------------------------------------
# 5. Evaluation Metrics Definition
# ---------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    macro_f1 = f1_score(labels, preds, average='macro')
    accuracy = accuracy_score(labels, preds)
    return {
        'macro_f1': macro_f1,
        'accuracy': accuracy
    }

# ---------------------------------------------------------
# 6. Model Initialization & TPU Training Arguments
# ---------------------------------------------------------
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=5
    )

training_args = TrainingArguments(
    output_dir='./tpu_results',
    num_train_epochs=6,
    per_device_train_batch_size=64,   # Larger batch size optimized for TPU MXUs
    per_device_eval_batch_size=128,
    bf16=True,                        # Native bfloat16 hardware acceleration on TPU v5e
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=SEED
)

trainer = CustomWeightedTrainer(
    class_weights=class_weights_tensor,
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

# ---------------------------------------------------------
# 7. Execute Training & Test Inference
# ---------------------------------------------------------
print("\n4. Starting TPU Training...")
trainer.train()

print("\n5. Running Test Set Predictions...")
raw_predictions = trainer.predict(tokenized_test)
predicted_classes = np.argmax(raw_predictions.predictions, axis=1)

# Convert 0-4 predictions back to 1-5 ratings scale
final_ratings = predicted_classes + 1

# ---------------------------------------------------------
# 8. Generate Submission File
# ---------------------------------------------------------
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'overall': final_ratings
})

submission.to_csv('submission.csv', index=False)
print("\nSuccess! TPU Execution complete and 'submission.csv' generated.")
print(submission.head())

Device: cuda
GPUs available for parallel processing: 1

1. Loading datasets...
2. Constructing structured prompts...
Calculated Class Weights (1-5 star penalty scale): [9.19 1.7  0.64 0.57 1.02]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



3. Starting Training for 12 Epochs (Batch Size: 32)...



Epoch 1/12: 100%|██████████| 645/645 [02:42<00:00,  3.96it/s]


---> Epoch 1 Stats | Train Loss: 1.4632 | Val Macro F1: 0.3296 | Val Acc: 0.3518
     [+] New Best Model saved! Macro F1: 0.3296



Epoch 2/12: 100%|██████████| 645/645 [02:41<00:00,  3.98it/s]


---> Epoch 2 Stats | Train Loss: 1.2357 | Val Macro F1: 0.2917 | Val Acc: 0.3683


Epoch 3/12:  42%|████▏     | 273/645 [01:08<01:33,  3.96it/s]


KeyboardInterrupt: 